In [ ]:
import jax
import jax.numpy as jnp
import jraph
import numpy as np
import time

from dataclasses import dataclass
from pprint import pprint

from so3lr.mlff.calculators.potential import load_model_from_workdir
from so3lr.mlff.utils import jraph_utils
from so3lr.mlff.data import AseDataLoaderSparse
from so3lr.mlff.data import NpzDataLoaderSparse

from so3lr import So3lr

In [ ]:
dataloader = NpzDataLoaderSparse(
    'data/ethanol.npz',
)

data, stats = dataloader.load(
    cutoff=4.5,
    cutoff_lr=np.inf,
    calculate_neighbors_lr=True,
    pick_idx=np.arange(50) # take the first 50 frames from the data set
)

In [ ]:
# The data loader takes the input data file and converts each structure into a `jraph.GraphsTuple`.

pprint(data[0])

In [ ]:
# Create a fresh iterator.

batch_size = 1

# Batch the graphs.
batched_graphs = jraph_utils.dynamically_batch_with_lr(
    data,
    n_node=stats['max_num_of_nodes'] * batch_size + 1,
    n_edge=stats['max_num_of_edges'] * batch_size + 1,
    n_graph=batch_size + 1,
    n_pairs=stats['max_num_of_pairs'] * batch_size + 1
)


so3lr_calc = So3lr(
    model='so3lr-tauri-m-2606',
    calculate_forces=True,
    lr_cutoff=100
)

processed_structures = 0
start = time.time()
for graph_batch, lr_batch in batched_graphs:
    # Transform the batched graph to inputs dict.
    inputs = jraph_utils.graph_to_batch_fn(
        graph_batch,
        lr_batch
    )

    output = jax.block_until_ready(so3lr_calc(inputs))
    processed_structures += int(
        np.asarray(jraph.get_graph_padding_mask(graph_batch)).sum()
    )

end = time.time()

assert processed_structures == len(data)
for key in ('forces', 'energy', 'dipole_vec', 'a0_ratios'):
    assert np.isfinite(np.asarray(output[key])).all(), key
print(f'Time without jit = {(end - start):.4g}')
print(f'Throughput without jit = {processed_structures / (end - start):.4g} structures/s')

In [ ]:
# Create a fresh iterator.

batch_size = 1

# Batch the graphs. We discuss below why this is necessary when running with jax.jit.
batched_graphs = jraph_utils.dynamically_batch_with_lr(
    data,
    n_node=stats['max_num_of_nodes'] * batch_size + 1,
    n_edge=stats['max_num_of_edges'] * batch_size + 1,
    n_graph=batch_size + 1,
    n_pairs=stats['max_num_of_pairs'] * batch_size + 1
)

so3lr_calc_jit = jax.jit(so3lr_calc)

i = 0
start = time.time()

processed_structures = 0

for graph_batch, lr_batch in batched_graphs:
    # Transform the batched graph to inputs dict.
    inputs = jraph_utils.graph_to_batch_fn(
        graph_batch,
        lr_batch
    )
    
    if i == 0:
        start_compile = time.time()
    
    # Waits for computation to finish for accurate time measurement
    output = jax.block_until_ready(so3lr_calc_jit(inputs))

    if i == 0:
        end_compile = time.time()
    
    graph_batch.nodes['forces_true'] = output['forces']
    graph_batch.nodes['energy_true'] = output['energy']
    graph_batch.nodes['dipole_vec_true'] = output['dipole_vec']
    graph_batch.nodes['a0_ratios_true'] = output['a0_ratios']
    processed_structures += int(
        np.asarray(jraph.get_graph_padding_mask(graph_batch)).sum()
    )

    # predicted.append(jraph.unbatch(graph_batch))
    
    i += 1
    
end = time.time()

assert processed_structures == len(data)
for key in ('forces', 'energy', 'dipole_vec', 'a0_ratios'):
    assert np.isfinite(np.asarray(output[key])).all(), key
print('Total number of iterations = ', i)
print(f'Time for jit compile = {(end_compile - start_compile):.4g}')
print(f'Time for jit evaluation = {(end - end_compile):.4g}')
print(f'Total time = {(end - start):.4g}')
print(f'JIT throughput = {processed_structures / (end - start):.4g} structures/s')

In [ ]:
# You can see the last entry has zeros, due to padding graph. See next section for explanation.
pprint(output)

# Why use jraph batching?

Why use `jraph.dynamically_batch` instead of constructing one `inputs` dictionary per structure? The first `jax.jit` call traces and compiles the computation with XLA, which can substantially accelerate later evaluations. However, a new input shape normally triggers another compilation. Consider a function and its jitted counterpart:

```python
fn = lambda A, B: do stuff ..
fn_jit = jax.jit(fn)
```

Changing the shapes of `A` or `B` can trigger recompilation.


For SO3LR structures of different sizes, the position arrays might have these shapes:

```python
inputs1['positions'].shape = (N1, 3)
inputs2['positions'].shape = (N2, 3)
inputs3['positions'].shape = (N3, 3)
```

where `N1`, `N2`, and `N3` are the atom counts. Padding the structure-dependent arrays to fixed limits avoids recompiling for each size. Given an iterator over `(jraph.GraphsTuple, long-range data)` pairs, `jraph_utils.dynamically_batch_with_lr` applies those limits through `n_node`, `n_edge`, `n_graph`, and `n_pairs`. A final padding graph receives contributions from padded nodes, edges, and long-range pairs. The example below batches two structures at a time and chooses the limits from the loader statistics.

In [ ]:
batch_size = 2

batched_graphs = jraph_utils.dynamically_batch_with_lr(
    data,
    n_node=stats['max_num_of_nodes'] * batch_size + 1,
    n_edge=stats['max_num_of_edges'] * batch_size + 1,
    n_graph=batch_size + 1,
    n_pairs=stats['max_num_of_pairs'] * batch_size + 1
)

# The arrays are numpy arrays at this point so they are on CPU and not on GPU.
example_batch, example_lr_batch = next(batched_graphs)
jax.tree.map(lambda x: print(type(x)), (example_batch, example_lr_batch))

In [ ]:
pprint(jraph_utils.batch_info_fn(example_batch))